## TASK 1. PROJECT OVERVIEW & KEY LEARNING OBJECTIVES

This is the **Gemini version** of the original OpenAI Agents notebook.  
Every cell is a line-by-line equivalent using **Google's free Gemini model** instead of OpenAI GPT.

## TASK 2. SETUP GEMINI AND TAVILY API KEYS

Instead of OpenAI, we use **Google's Gemini API** (free tier available).

- **Gemini API Key:** Go to [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey) and create a key.
- **Tavily API Key:** Go to [https://tavily.com/](https://tavily.com/) and sign up for a free key.

Create a `.env` file in the same directory as this notebook:
```
GEMINI_API_KEY=your-gemini-api-key-here
TAVILY_API_KEY=tvly-YourSecretTavilyKey...
```

### Quick Reference: OpenAI → Gemini Mapping

| Original (OpenAI) | Gemini Equivalent |
|---|---|
| `pip install openai-agents` | `pip install google-generativeai` |
| `from openai import OpenAI` | `import google.generativeai as genai` |
| `OpenAI(api_key=...)` | `genai.configure(api_key=...)` |
| `OPENAI_API_KEY` | `GEMINI_API_KEY` |
| `model="gpt-4.1-mini"` | `model_name="gemini-2.0-flash"` |
| `Agent(name, instructions, model, tools)` | `genai.GenerativeModel(model_name, tools, system_instruction)` |
| `@function_tool` decorator | Plain Python function (Gemini auto-generates schema) |
| `params: TypedDict` | Direct keyword arguments |
| `SQLiteSession` (memory) | `chat.history` list (in-memory) |
| `await Runner.run(agent, input, session)` | `chat.send_message()` + function call loop |
| `CodeInterpreterTool` | `"code_execution"` (Gemini built-in) |
| `FunctionTool` | Just pass the function in `tools=[]` |

In [ ]:
# ── Original Cell 5: pip install ──────────────────────────────────────────
# ORIGINAL:  %pip install -q openai-agents==0.2.2 python-dotenv requests
# GEMINI:    Replace openai + openai-agents with google-generativeai

%pip install -q google-generativeai python-dotenv requests

In [ ]:
# ── Original Cell 6: Load keys & create client ──────────────────────────────
# ORIGINAL:
#   import os
#   import requests
#   from openai import OpenAI
#   from dotenv import load_dotenv
#   from IPython.display import display, Markdown
#   load_dotenv()
#   openai_api_key = os.getenv("OPENAI_API_KEY")
#   tavily_api_key = os.getenv("TAVILY_API_KEY")
#   openai_client = OpenAI(api_key=openai_api_key)
#
# GEMINI EQUIVALENT:

import os
import json
import requests
from dotenv import load_dotenv
from IPython.display import display, Markdown
import google.generativeai as genai          # Replaces: from openai import OpenAI

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")  # Replaces: os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

print("✅ Keys loaded:")
print(f"Gemini API Key: {gemini_api_key[:5]}***")
print(f"Tavily API Key: {tavily_api_key[:5]}***")

# Configure the Gemini SDK (replaces: openai_client = OpenAI(api_key=...))
genai.configure(api_key=gemini_api_key)

In [ ]:
# ── Original Cell 7: Helper to print markdown ───────────────────────────────
# ORIGINAL:
#   def print_markdown(text: str):
#       display(Markdown(text))
#
# GEMINI EQUIVALENT: (identical — this has nothing to do with the model)

def print_markdown(text: str):
    display(Markdown(text))

## TASK 3: DEFINE TAVILY SEARCH FUNCTION & CREATE A TOOL

In [ ]:
# ── Original Cell 9: Imports ─────────────────────────────────────────────────
# ORIGINAL:
#   import json
#   from typing_extensions import TypedDict
#   from agents import function_tool
#
# GEMINI EQUIVALENT:
#   - We don't need the `function_tool` decorator from OpenAI Agents SDK.
#   - Gemini has its own function-calling mechanism.
#   - We define a plain Python function and pass it in the tools list.

import json
from typing_extensions import TypedDict
# NOTE: No equivalent of "from agents import function_tool" is needed.
# Gemini auto-generates the tool schema from the function signature & docstring.

In [ ]:
# ── Original Cell 10: Define param type ──────────────────────────────────────
# ORIGINAL:
#   class TavilySearchParams(TypedDict):
#       query: str
#       max_results: int
#
# GEMINI EQUIVALENT:
#   Gemini function calling passes arguments directly as keyword args,
#   so we don't strictly need a TypedDict. We keep it here for reference.

class TavilySearchParams(TypedDict):
    query: str         # The search query string
    max_results: int   # The maximum number of results to return

In [ ]:
# ── Original Cell 11: Tavily search function ─────────────────────────────────
# ORIGINAL:
#   @function_tool                              <-- OpenAI decorator (REMOVED)
#   def tavily_search(params: TavilySearchParams) -> str:
#       payload = {
#           "query": params["query"],          <-- accessed via dict key
#           "max_results": params.get("max_results", 2),
#       }
#
# GEMINI EQUIVALENT:
#   - No decorator needed — Gemini reads the function signature + docstring.
#   - Parameters are direct keyword args instead of a TypedDict dict.

def tavily_search(query: str, max_results: int = 2) -> str:
    """
    Calls the Tavily API and returns a string summary of top search results.

    Args:
        query (str): The search query string.
        max_results (int): The maximum number of results to return (default 2).

    Returns:
        str: A formatted string summarizing the top search results, or an error message.
    """

    # The web address (endpoint) for sending search requests to Tavily
    url = "https://api.tavily.com/search"

    # Tell the API that we're sending JSON data
    headers = {"Content-Type": "application/json"}

    # What we're sending to the API:
    payload = {
        "api_key": tavily_api_key,     # Our Tavily API key from .env
        "query": query,                 # CHANGED: was params["query"]
        "max_results": max_results,     # CHANGED: was params.get("max_results", 2)
    }

    # Send the search request to Tavily
    response = requests.post(url, json=payload, headers=headers)

    # Check if the search worked (200 = OK)
    if response.status_code == 200:
        results = response.json().get("results", [])

        # Build a summary string with each result's title and content, numbered
        summary = "\n".join(
            [f"{i+1}. {r['title']}: {r['content']}" for i, r in enumerate(results)]
        )
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"


print("✅ tavily_search function defined.")

**PRACTICE OPPORTUNITY:**
- **Modify the tavily_search function so that it returns only the titles of the search results, without including the content.**
- **Increase the number of results returned from 2 to 3 by updating the max_results parameter in the function.**

## TASK 4: BUILD AND RUN AN AI AGENT WITH SEARCH TOOL

In [ ]:
# ── Original Cell 15: Session / Memory ───────────────────────────────────────
# ORIGINAL:
#   from agents import SQLiteSession
#   session = SQLiteSession("live_researcher_practice")
#
# GEMINI EQUIVALENT:
#   Gemini's ChatSession keeps conversation history in memory automatically.
#   We maintain a simple list that acts as the conversation history.
#   For persistent storage across restarts, you'd add your own SQLite logic.

conversation_history = []  # Replaces SQLiteSession — stores chat turns in memory
print("✅ Conversation history initialized (replaces SQLiteSession).")

In [ ]:
# ── Original Cell 16: Create the Agent ───────────────────────────────────────
# ORIGINAL:
#   from agents import Agent
#   live_researcher_agent = Agent(
#       name="Live Market Researcher",
#       instructions="..."  ,
#       model="gpt-4.1-mini",
#       tools=[tavily_search],
#   )
#
# GEMINI EQUIVALENT:
#   - Agent()            → genai.GenerativeModel()
#   - name=              → (not needed in Gemini, optional in system prompt)
#   - instructions=      → system_instruction=
#   - model="gpt-4.1-mini" → model_name="gemini-2.0-flash" (free tier)
#   - tools=[...]        → tools=[...]  (same concept, Gemini auto-generates schema)

RESEARCHER_SYSTEM_INSTRUCTION = """
CONTEXT:
You are a world-class market research assistant with access to real-time web search via the tavily_search tool.

INSTRUCTION:
- Analyze the user's question and determine if recent or real-time information is needed.
- If the question involves recent events, news, or product info, always call tavily_search.
- Summarize search results clearly and concisely, do not copy-paste.
- Always start your answer with: "🔍 According to a web search …"

INPUT:
You will receive a conversation history and the latest user question. Use the full context to inform your response.

OUTPUT:
Provide a clear, well-structured answer that references the search results when appropriate. If you use tavily_search, integrate the findings into your summary.
"""

live_researcher_agent = genai.GenerativeModel(
    model_name="gemini-2.0-flash",             # FREE model (replaces gpt-4.1-mini)
    tools=[tavily_search],                      # Register tavily_search as a callable tool
    system_instruction=RESEARCHER_SYSTEM_INSTRUCTION,  # Replaces instructions=
)

print("✅ Agent created with Tavily tool.")

In [ ]:
# ── Helper: Run agent with automatic function calling ────────────────────────
# OpenAI Agents SDK's Runner.run() automatically handles the tool-call loop.
# With Gemini, we handle this loop ourselves.  This helper function:
#   1. Sends the user message to Gemini
#   2. If Gemini wants to call a function, we execute it and send the result back
#   3. Repeats until Gemini returns a final text answer
#
# This replaces:  await Runner.run(starting_agent=..., input=..., session=...)

def run_agent(model, user_input: str, history: list) -> str:
    """
    Runs a Gemini agent with automatic function-calling loop.
    Equivalent to: await Runner.run(agent, input, session)

    Args:
        model: The Gemini GenerativeModel with tools configured.
        user_input: The user's question.
        history: Conversation history list (modified in place for memory).

    Returns:
        str: The agent's final text response.
    """
    # Start or continue a chat session with history for memory
    chat = model.start_chat(history=history)

    # Send the user's message
    response = chat.send_message(user_input)

    # Loop: handle function calls until we get a text response
    max_iterations = 10  # Safety limit
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        part = response.candidates[0].content.parts[0]

        # Check if the model wants to call a function
        if hasattr(part, "function_call") and part.function_call.name:
            function_call = part.function_call
            function_name = function_call.name
            function_args = dict(function_call.args)

            print(f"  🔧 Agent calling tool: {function_name}({function_args})")

            # Execute the function
            if function_name == "tavily_search":
                result = tavily_search(**function_args)
            else:
                result = f"Unknown function: {function_name}"

            # Send the function result back to Gemini
            response = chat.send_message(
                genai.protos.Content(
                    parts=[
                        genai.protos.Part(
                            function_response=genai.protos.FunctionResponse(
                                name=function_name,
                                response={"result": result},
                            )
                        )
                    ]
                )
            )
        else:
            # We got a text response — break out of the loop
            break

    # Extract the final text answer
    final_text = response.candidates[0].content.parts[0].text

    # Update conversation history for memory (replaces SQLiteSession)
    history.extend(chat.history)

    return final_text


print("✅ run_agent() helper defined (replaces Runner.run).")

In [ ]:
# ── Original Cell 17: First question ─────────────────────────────────────────
# ORIGINAL:
#   from agents import Runner
#   q1 = "What are people saying about the new GPT-5.4 Model?"
#   run1 = await Runner.run(
#       starting_agent=live_researcher_agent, input=q1, session=session)
#   print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")
#
# GEMINI EQUIVALENT:

q1 = "What are people saying about the new GPT-5.4 Model?"

print_markdown(f"**User:** {q1}")

# Replaces: await Runner.run(starting_agent=live_researcher_agent, input=q1, session=session)
answer1 = run_agent(live_researcher_agent, q1, conversation_history)

# Replaces: run1.final_output
print_markdown(f"### 🤖 Agent's Answer\n{answer1}")

- **Note:** You can monitor Gemini API usage through [Google AI Studio](https://aistudio.google.com/).
- Unlike OpenAI's trace logs, Gemini function calls are visible in the `chat.history` object and printed by our `run_agent()` helper.

**PRACTICE OPPORTUNITY:**  
- **Use the `live_researcher_agent` AI agent to find and summarize the key features of your favorite vehicle.**  
    - **Example:** *"What do reviewers say about the new Cybertruck?"*  
    - **Then test its memory and search ability with a follow-up:** *"Summarize the main features."*  

## TASK 5. LEVERAGE EXISTING BUILT-IN TOOLS

Instead of OpenAI's built-in tools, Gemini provides its own:
- **`code_execution`** — Built-in Python code execution (replaces OpenAI's `CodeInterpreterTool`)
- **`google_search`** — Built-in Google Search (replaces OpenAI's `WebSearchTool`)
- **Custom functions** — Any Python function passed in `tools=[]` (replaces `FunctionTool`)

### ⚠️ Gemini Limitation
**Gemini does NOT allow combining built-in tools (`code_execution`) with custom function calling (`tavily_search`) in the same model.**  
This is a known API restriction: `Built-in tools and Function Calling cannot be combined in the same request.`

**Solution:** We create **two separate agents** — one for search, one for code — and an **orchestrator** that chains them together (search first → code second). This achieves the same result as the original OpenAI notebook.

In [ ]:
# ── Original Cell 24: Code Interpreter tool ──────────────────────────────────
# ORIGINAL:
#   from agents import FunctionTool, function_tool, CodeInterpreterTool
#   code_interpreter = CodeInterpreterTool(
#       tool_config={"type": "code_interpreter",
#                    "container": {"type": "auto"}})
#
# GEMINI EQUIVALENT:
#   Gemini 2.0 Flash supports "code_execution" as a built-in tool.
#   However, it CANNOT be combined with custom function calling (tavily_search)
#   in the same model. So we create a SEPARATE code execution agent.

# Agent 1: Search Agent (already created above as live_researcher_agent)
#   → Uses tavily_search (custom function calling)

# Agent 2: Code Execution Agent (new)
#   → Uses code_execution (built-in tool)

code_agent = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    tools=["code_execution"],           # Built-in code execution ONLY
    system_instruction="""
You are a Python data analyst. You receive data or context from a prior web search
and your job is to perform calculations, analysis, or simulations using Python code.
Use code_execution to write and run Python code. Explain your process and results clearly.
""",
)

print("✅ Tool ready: code_agent (Gemini built-in code_execution)")
print("✅ Tool ready: live_researcher_agent (tavily_search)")

In [ ]:
# ── Original Cell 25: Create Analyst Agent with both tools ───────────────────
# ORIGINAL:
#   analyst_agent = Agent(
#       name="Analyst Agent",
#       instructions="...",
#       model="gpt-4.1-mini",
#       tools=[tavily_search, code_interpreter],
#   )
#
# GEMINI EQUIVALENT:
#   Since Gemini can't combine built-in tools + function calling,
#   we DON'T create a single analyst_agent.
#   Instead, we use an ORCHESTRATOR pattern:
#     Step 1: search_agent  (tavily_search)   → gets real-time data
#     Step 2: code_agent    (code_execution)   → does calculations on that data
#     Step 3: combine results into final answer

ANALYST_SYSTEM_INSTRUCTION = """
CONTEXT:
You are a world-class market research assistant with access to both real-time web search (via the tavily_search tool) and Python code execution (via the built-in code_execution tool).

INSTRUCTION:
- Carefully analyze the user's question to determine whether it requires:
    - Recent or real-time information (use tavily_search),
    - Data analysis, calculations, or code execution (use code_execution),
    - Or a combination of both tools.
- Use tavily_search for up-to-date facts, news, or product information.
- Use code_execution for tasks involving data analysis, calculations, or code-based reasoning.
- If the task benefits from both tools, use them together and integrate the results.
- Clearly summarize your findings and reasoning. Do not copy-paste search results; always paraphrase.
- When using search, begin your answer with: \"🔍 According to a web search …\"
- When using code, explain your process and results clearly.

INPUT:
You will receive a conversation history and the latest user question. Use the full context to decide which tool(s) to use and to inform your response.

OUTPUT:
Provide a clear, well-structured answer. Reference search results and/or code outputs as appropriate, and integrate them into your summary.
"""

# NOTE: We keep the system instruction above for reference, but it is now
# split across two agents. The orchestrator function below handles the routing.

print("✅ Analyst orchestrator ready (2-agent pattern: search → code).")

In [ ]:
# ── Orchestrator: Chains search agent → code agent ───────────────────────────
# This replaces the single analyst_agent + Runner.run() from the original.
# It works in two phases:
#   Phase 1: Use the search agent (tavily_search) to find real-time data
#   Phase 2: Feed search results to the code agent (code_execution) for analysis

def run_analyst_agent(user_input: str) -> str:
    """
    Orchestrator that chains search + code execution agents.
    Equivalent to: await Runner.run(analyst_agent, input, session)
    where analyst_agent had both tavily_search and code_interpreter.

    Args:
        user_input: The user's question.

    Returns:
        str: The combined final answer.
    """
    # ── PHASE 1: Web Search ──────────────────────────────────────────────────
    print("📡 Phase 1: Searching the web for real-time data...")
    search_history = []
    search_result = run_agent(live_researcher_agent, user_input, search_history)
    print(f"  ✅ Search complete.\n")

    # ── PHASE 2: Code Execution ──────────────────────────────────────────────
    print("💻 Phase 2: Running calculations with code execution...")
    code_prompt = (
        f"Based on the following web search results:\n\n"
        f"{search_result}\n\n"
        f"Now do the following using Python code:\n"
        f"{user_input}\n\n"
        f"Use the data from the search results above. Write and execute Python code "
        f"to perform the calculations, then provide a clear summary of all results."
    )

    code_chat = code_agent.start_chat()
    code_response = code_chat.send_message(code_prompt)

    # Extract all text from the code agent's response
    final_text = ""
    for part in code_response.candidates[0].content.parts:
        if hasattr(part, "text") and part.text:
            final_text += part.text

    print(f"  ✅ Code execution complete.\n")

    return final_text


print("✅ run_analyst_agent() orchestrator defined.")

In [ ]:
# ── Original Cell 26: Analyst Agent question ─────────────────────────────────
# ORIGINAL:
#   session = SQLiteSession("live_researcher")
#   q1 = ("Find the current price of a Tesla Cyber Truck in Canada. "
#         "Then, simulate how the price would change if import tariffs "
#         "increased from 5% to 20% in 1% increments. ...")
#   run1 = await Runner.run(starting_agent=analyst_agent, input=q1, session=session)
#   print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")
#
# GEMINI EQUIVALENT:

q_analyst = (
    "Find the current price of a Tesla Cyber Truck in Canada. "
    "Then, simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments. "
    "For each tariff rate, calculate the new price using Python and provide a summary of the results across all scenarios."
)

print_markdown(f"**User:** {q_analyst}")

# Replaces: await Runner.run(starting_agent=analyst_agent, input=q1, session=session)
analyst_answer = run_analyst_agent(q_analyst)

# Replaces: run1.final_output
print_markdown(f"### 🤖 Agent's Answer\n{analyst_answer}")

# PRACTICE OPPORTUNITY SOLUTIONS

**PRACTICE OPPORTUNITY SOLUTION:**
- **Modify the tavily_search function so that it returns only the titles of the search results, without including the content.**
- **Increase the number of results returned from 2 to 3 by updating the max_results parameter in the function.**

In [ ]:
# ── Original Cell 29: Modified tavily_search (titles only) ────────────────────
# ORIGINAL:
#   @function_tool
#   def tavily_search(params: TavilySearchParams) -> str:
#       ...
#       summary = "\n".join([f"{i+1}. {r['title']}" for i, r in enumerate(results)])
#       ...
#       payload = { "max_results": params.get("max_results", 3) }
#
# GEMINI EQUIVALENT:

def tavily_search_titles_only(query: str, max_results: int = 3) -> str:
    """
    Calls the Tavily API and returns only the titles from the top search results.

    Args:
        query (str): The search query string.
        max_results (int): The maximum number of results to return (default 3).

    Returns:
        str: Numbered list of result titles, or an error message.
    """
    url = "https://api.tavily.com/search"
    headers = {"Content-Type": "application/json"}
    payload = {
        "api_key": tavily_api_key,
        "query": query,
        "max_results": max_results,  # Default 3 results instead of 2
    }

    response = requests.post(url, json=payload, headers=headers)

    if response.status_code == 200:
        results = response.json().get("results", [])

        # Only return numbered titles (no content)
        summary = "\n".join([f"{i+1}. {r['title']}" for i, r in enumerate(results)])

        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"


print("✅ tavily_search_titles_only function defined (practice solution).")

**PRACTICE OPPORTUNITY SOLUTION:**  
- **Use the `live_researcher_agent` AI agent to find and summarize the key features of your favorite vehicle.**  
    - **Example:** *"What do reviewers say about the new Cybertruck?"*  
    - **Then test its memory and search ability with a follow-up:** *"Summarize the main features."*  

In [ ]:
# ── Original Cell 31: Practice — question 1 ──────────────────────────────────
# ORIGINAL:
#   from agents import Runner
#   session = SQLiteSession("live_researcher_practice")
#   q1 = "What do reviewers say about the new Tesla Cybertruck?"
#   run1 = await Runner.run(live_researcher_agent, q1, session=session)
#
# GEMINI EQUIVALENT:

practice_history = []  # Fresh history for this practice session

pq1 = "What do reviewers say about the new Tesla Cybertruck?"
print_markdown(f"**User:** {pq1}")

panswer1 = run_agent(live_researcher_agent, pq1, practice_history)
print_markdown(f"**Answer:** {panswer1}")

In [ ]:
# ── Original Cell 32: Practice — follow-up (tests memory) ───────────────────
# ORIGINAL:
#   q2 = "Summarize the main features in one paragraph"
#   run2 = await Runner.run(live_researcher_agent, q2, session=session)
#
# GEMINI EQUIVALENT:

pq2 = "Summarize the main features in one paragraph"
print_markdown(f"**User:** {pq2}")

panswer2 = run_agent(live_researcher_agent, pq2, practice_history)
print_markdown(f"**Answer:** {panswer2}")